# Tu primer agente: el analista de la Liga Intercolegial ⚽

Imagina que tu colegio juega una liga de fútbol contra otros cinco colegios. Van cuatro fechas, falta una, y todos quieren saber lo mismo: ¿cómo va la tabla?, ¿quién es el goleador?, ¿qué necesita mi equipo para salir campeón?

En este notebook vas a construir un **agente de inteligencia artificial** que responde esas preguntas. No le vas a dar las respuestas: le vas a dar los resultados de los partidos, y él solo va a hacer las cuentas, consultarte lo que le falte y escribir el análisis.

## ¿Qué es un agente?

Un chatbot normal solo **escribe texto**. Un agente además puede **hacer cosas**: abrir archivos, escribir y ejecutar programas, pedir información. Y lo hace en un ciclo, como una persona resolviendo un problema:

> pienso qué necesito → uso una herramienta → miro el resultado → pienso otra vez → … → respondo

## Las cuatro piezas, en idioma de fútbol

| Pieza | En fútbol sería… | Qué es en realidad |
|---|---|---|
| **Agente** | El jugador que fichaste y las instrucciones que le dio el técnico | El modelo de IA, sus instrucciones y sus herramientas |
| **Entorno** | La cancha donde juega | Una computadora en la nube donde el agente ejecuta programas |
| **Sesión** | Un partido concreto | Una conversación concreta con el agente |
| **Eventos** | Las jugadas, narradas minuto a minuto | Cada mensaje y cada acción que ocurre en la conversación |

Al jugador lo fichas una vez y juega muchos partidos. Igual aquí: el agente y el entorno se crean una vez, y después puedes abrir todas las sesiones que quieras.

## El plan del partido

| Paso | Qué haces | Sección |
|---|---|---|
| **A. Preparar los datos** | Escribes los resultados del torneo y los guardas en archivos | 1 |
| **B. Diseñar el agente** | Le escribes sus instrucciones y le declaras una herramienta | 2 |
| **C. Crear el agente** | Lo registras en Claude Platform | 3 |
| **D. Crear el entorno** | Le das una computadora en la nube | 4 |
| **E. Subir los archivos** | Le mandas los resultados | 5 |
| **F. Abrir la sesión** | Empieza el partido | 6 |
| **G. Programar tu herramienta** | Escribes la función que el agente te podrá pedir | 7 |
| **H. Conversar** | Le haces preguntas y ves cómo trabaja | 8 |
| **I. Revisar y limpiar** | Compruebas si acertó y borras la sesión | 9 y 10 |

**Cómo usar el notebook:** ejecuta las celdas de una en una y en orden (Shift+Enter). Lee lo que imprime cada una antes de seguir.

> Las celdas marcadas con ☁️ usan la API de Claude: crean cosas reales en la cuenta y las de conversación tienen un costo pequeño. Las demás corren solo en tu computadora.

## 0. Calentamiento: conectarse a Claude

Para usar Claude desde un programa necesitas una **API key**: una contraseña larga que identifica tu cuenta. Nunca se escribe dentro del código ni se comparte. Dónde se guarda depende de dónde estés ejecutando el notebook:

- **En Google Colab:** abre el panel **Secretos** (el ícono de llave 🔑 a la izquierda), crea un secreto llamado `ANTHROPIC_API_KEY`, pega tu key como valor y activa "Acceso desde el notebook".
- **En tu computadora:** crea un archivo `.env` en esta carpeta con la línea `ANTHROPIC_API_KEY=sk-ant-...tu-key...`. Necesitas tener instalados los paquetes `anthropic` y `python-dotenv`.

La celda detecta sola en cuál de los dos estás. En Colab además instala el paquete `anthropic`. Solo comprueba que la key existe; no la muestra.

In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    %pip install -q anthropic
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv(override=True)

key = os.environ.get("ANTHROPIC_API_KEY", "")
assert key and key != "sk-ant-...", "Falta la API key: revisa las instrucciones de arriba y vuelve a ejecutar esta celda"
print("API key cargada ✔")

Ahora creamos el **cliente**: el objeto con el que le hablamos a Claude Platform. Todo lo que hagamos después pasa por él.

Para probar que la conexión funciona le pedimos la lista de modelos disponibles. Esta consulta no cuesta nada.

In [ ]:
import anthropic

client = anthropic.Anthropic()

for m in client.models.list(limit=5).data:
    print(m.id.ljust(32), m.display_name.ljust(22), m.created_at.date(), m.max_input_tokens, m.max_tokens)


## 1. Los datos del torneo (sin API)

Un agente es tan bueno como la información que le das. Empezamos por ahí.

La **Liga Intercolegial** tiene seis equipos y juegan todos contra todos una vez: cinco fechas, tres partidos por fecha. Ya se jugaron cuatro fechas.

### 1.1 Los resultados

Cada partido es una fila: fecha, equipo local, goles del local, goles del visitante, equipo visitante.

In [ ]:
partidos = [
    # fecha, local, goles_local, goles_visitante, visitante
    (1, "Halcones",  2, 1, "Delfines"),
    (1, "Tiburones", 1, 1, "Jaguares"),
    (1, "Leones",    3, 0, "Cóndores"),
    (2, "Halcones",  0, 2, "Jaguares"),
    (2, "Delfines",  2, 2, "Cóndores"),
    (2, "Tiburones", 1, 0, "Leones"),
    (3, "Halcones",  3, 1, "Cóndores"),
    (3, "Jaguares",  1, 2, "Leones"),
    (3, "Delfines",  0, 1, "Tiburones"),
    (4, "Halcones",  1, 1, "Leones"),
    (4, "Cóndores",  2, 3, "Tiburones"),
    (4, "Jaguares",  4, 0, "Delfines"),
]

print(len(partidos), "partidos jugados")
print(sum(gl + gv for _, _, gl, gv, _ in partidos), "goles en total")

### 1.2 Los goles

Para saber quién es el goleador necesitamos el detalle de cada gol: quién lo hizo y en qué minuto.

Cada equipo tiene cuatro jugadores que suelen anotar. En lugar de escribir los 34 goles a mano, dejamos que el computador los reparta al azar entre ellos, respetando el marcador de cada partido. `random.Random(7)` fija la "suerte": siempre sale el mismo reparto, así todos en el salón tenemos los mismos datos.

In [ ]:
import random

plantillas = {
    "Halcones":  ["Mateo Rojas", "Valentina Cruz", "Samuel Peña", "Lucía Mora"],
    "Tiburones": ["Isabella Gil", "Tomás Vega", "Emilio Paz", "Sara Luna"],
    "Leones":    ["Daniel Soto", "Camila Ríos", "Andrés Lara", "Mariana Díaz"],
    "Jaguares":  ["Sofía Marín", "Julián Cano", "Gabriela León", "Nicolás Rey"],
    "Delfines":  ["Martín Sáenz", "Paula Arias", "Felipe Cortés", "Elena Bravo"],
    "Cóndores":  ["Antonia Mejía", "David Parra", "Luciana Toro", "Simón Vidal"],
}

suerte = random.Random(7)
goles = []
for fecha, local, gl, gv, visitante in partidos:
    for equipo, cantidad in ((local, gl), (visitante, gv)):
        for _ in range(cantidad):
            jugador = suerte.choices(plantillas[equipo], weights=[5, 3, 2, 1])[0]
            minuto = suerte.randint(1, 90)
            goles.append((fecha, equipo, jugador, minuto))

print(len(goles), "goles registrados. Los primeros cinco:")
for g in goles[:5]:
    print(g)

### 1.3 Guardarlos en archivos

El agente va a trabajar en otra computadora, en la nube, así que hay que mandarle los datos como archivos. Usamos el formato **CSV**: texto simple, una fila por línea y los valores separados por comas. Es el formato que abre cualquier hoja de cálculo.

In [ ]:
import csv
from pathlib import Path

DATA = Path("data")
DATA.mkdir(exist_ok=True)

with open(DATA / "partidos.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["fecha", "local", "goles_local", "goles_visitante", "visitante"])
    w.writerows(partidos)

with open(DATA / "goles.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["fecha", "equipo", "jugador", "minuto"])
    w.writerows(goles)

print((DATA / "partidos.csv").read_text(encoding="utf-8"))

### 1.4 El calendario: un dato que NO vamos a subir

Falta la quinta fecha. El calendario lo dejamos **solo en tu computadora**, a propósito. Más adelante el agente lo va a necesitar y tendrá que pedírtelo: así vas a ver cómo un agente en la nube puede consultar información que solo tú tienes.

In [ ]:
calendario = [
    {"fecha": 5, "local": "Halcones", "visitante": "Tiburones", "dia": "sábado", "hora": "9:00",  "cancha": "Colegio San Marcos"},
    {"fecha": 5, "local": "Leones",   "visitante": "Delfines",  "dia": "sábado", "hora": "11:00", "cancha": "Polideportivo Norte"},
    {"fecha": 5, "local": "Cóndores", "visitante": "Jaguares",  "dia": "sábado", "hora": "11:00", "cancha": "Colegio La Colina"},
]

for p in calendario:
    print(f'{p["local"]} vs {p["visitante"]} · {p["dia"]} {p["hora"]} · {p["cancha"]}')

## 2. Diseñar el agente (sin API)

Ahora decidimos cómo será el agente. Son dos decisiones: **qué instrucciones** tiene y **qué herramientas** puede usar.

### 2.1 Las instrucciones (el *system prompt*)

Es la charla técnica antes del partido: un texto que el agente lee antes de cualquier pregunta tuya. Un buen system prompt responde tres cosas:

- **¿Quién eres?** El analista de la liga.
- **¿Qué tienes a mano?** Dos archivos con resultados y goles, y una herramienta para el calendario.
- **¿Cómo debes trabajar?** Con las reglas del torneo, calculando con programas en lugar de adivinar, y explicando claro.

**Ojo con las dos carpetas.** El system prompt le habla *al agente*, así que describe la computadora *del agente*, no la tuya:

| | Dónde está | Carpeta |
|---|---|---|
| Tus archivos originales | Tu computadora (o Colab) | `data/` |
| La copia que usa el agente | Su computadora en la nube | `/mnt/session/uploads/` |

El agente no puede ver tu carpeta `data/`. Los archivos llegan a su carpeta en dos pasos: los subimos en la sección 5 y la sesión los coloca allí en la sección 6.

Fíjate en que las reglas de la liga (cuántos puntos da una victoria, cómo se desempata) van aquí. El agente no las puede adivinar: cada torneo tiene las suyas.

In [ ]:
SYSTEM = """\
Eres el analista deportivo de la Liga Intercolegial de fútbol. Hablas con
estudiantes de bachillerato: explica claro, en español, y con entusiasmo
pero sin exagerar.

Datos disponibles en tu sandbox (la computadora en la nube donde ejecutas
programas), en la carpeta /mnt/session/uploads/:
- partidos.csv: los partidos ya jugados (fecha, local, goles_local, goles_visitante, visitante)
- goles.csv: cada gol anotado (fecha, equipo, jugador, minuto)

El calendario de la próxima fecha NO está en esos archivos. Para consultarlo
usa la herramienta get_proximo_partido.

Reglas de la liga: victoria 3 puntos, empate 1, derrota 0. Los empates en
puntos se deciden por diferencia de gol y, si sigue el empate, por goles a favor.

Para cualquier cuenta escribe y ejecuta un programa de Python sobre los
archivos; nunca calcules de memoria ni inventes datos. Muestra las tablas
en formato de tabla.
"""

print(SYSTEM)

### 2.2 Las herramientas

Una herramienta es algo que el agente puede **hacer** además de escribir. Le damos de dos tipos:

**Las que ya vienen listas.** El paquete `agent_toolset_20260401` le permite ejecutar comandos, leer archivos y correr programas. Todo eso pasa **en la nube**, en la computadora del agente. El paquete trae ocho herramientas:

| Herramienta | Qué hace |
|---|---|
| `bash` | Ejecuta comandos en la terminal; con ella corre los programas de Python |
| `read` | Lee un archivo |
| `write` | Crea o sobrescribe un archivo |
| `edit` | Modifica una parte de un archivo |
| `glob` | Busca archivos por nombre, por ejemplo `*.csv` |
| `grep` | Busca un texto dentro de los archivos |
| `web_fetch` | Descarga una página web |
| `web_search` | Busca en internet |

Con `"default_config": {"enabled": True}` quedan activadas las ocho.

In [ ]:
herramientas_integradas = {
    "type": "agent_toolset_20260401",
    "default_config": {"enabled": True},
}

**La que inventamos nosotros.** Se llama `get_proximo_partido` y sirve para consultar el calendario. Aquí solo la *declaramos*, como quien anota una jugada en la pizarra; el código que la ejecuta lo escribimos en la sección 7 y correrá **en tu computadora**.

La declaración tiene tres partes:

- `name`: el nombre con el que el agente la va a pedir.
- `description`: para qué sirve. El agente decide cuándo usarla leyendo esto, así que debe ser claro.
- `input_schema`: qué datos hay que pasarle. Aquí uno solo, obligatorio: el nombre del equipo, que es un texto (`string`).

In [ ]:
get_proximo_partido_tool = {
    "type": "custom",
    "name": "get_proximo_partido",
    "description": "Devuelve el próximo partido de un equipo de la liga: rival, día, hora y cancha.",
    "input_schema": {
        "type": "object",
        "properties": {
            "equipo": {"type": "string", "description": "Nombre del equipo, por ejemplo Halcones"},
        },
        "required": ["equipo"],
    },
}

TOOLS = [herramientas_integradas, get_proximo_partido_tool]

## 3. Crear el agente ☁️

Este es el fichaje. Le decimos a Claude Platform:

- `name`: cómo se llama el agente.
- `model`: qué modelo de Claude es el "cerebro".
- `system`: las instrucciones de la sección 2.1.
- `tools`: las herramientas de la sección 2.2.

Todavía no pasa nada emocionante: el agente queda registrado y nos devuelven su **id**, un código que empieza por `agent_`. Con ese id lo podremos usar en todos los partidos que queramos.

In [ ]:
MODELO = "claude-opus-5-5"

agent = client.beta.agents.create(
    name="Analista de la Liga Intercolegial",
    model=MODELO,
    system=SYSTEM,
    tools=TOOLS,
)

agent_id = agent.id
print("agent_id:", agent_id)

### 3.1 Revisar el fichaje ☁️

¿Cómo quedó registrado el agente? Se lo preguntamos a Claude Platform con su id.

Para el paquete integrado, Claude Platform guarda dos cosas:

- `default_config`: la regla general para las ocho herramientas (aquí, todas activas).
- `configs`: solo las **excepciones** a esa regla. Como no pusimos ninguna, esa lista está vacía.

Claude Platform no devuelve los nombres de las ocho, así que la celda parte de la lista `OCHO` y le quita las que aparezcan apagadas en las excepciones. Lo que queda son las herramientas que el agente puede usar.

In [ ]:
OCHO = ["bash", "read", "write", "edit", "glob", "grep", "web_fetch", "web_search"]

agente = client.beta.agents.retrieve(agent_id)

for t in agente.tools:
    if t.type == "agent_toolset_20260401":
        apagadas = [c.name for c in t.configs if not c.enabled]
        print("Herramientas integradas activas:", [n for n in OCHO if n not in apagadas])
        print("Herramientas integradas apagadas:", apagadas or "ninguna")
    else:
        print("Herramienta propia:", t.name)

## 4. Crear el entorno ☁️

El entorno es la cancha: describe la computadora en la nube donde el agente va a ejecutar sus programas.

- `"type": "cloud"` significa que Anthropic la administra; tú no instalas nada.
- `"networking": {"type": "unrestricted"}` le permite conectarse a internet.

El nombre lleva unas letras al azar al final para que no se repita si ejecutas la celda dos veces.

In [ ]:
import uuid

env = client.beta.environments.create(
    name=f"liga-intercolegial-{uuid.uuid4().hex[:6]}",
    config={"type": "cloud", "networking": {"type": "unrestricted"}},
)

env_id = env.id
print("env_id:", env_id)

## 5. Subir los archivos ☁️

Mandamos los dos CSV a Claude Platform. Cada archivo subido recibe su propio id; los guardamos en un diccionario para usarlos en el siguiente paso.

El calendario se queda en casa, como acordamos.

In [ ]:
archivos = {}
for nombre in ["partidos.csv", "goles.csv"]:
    with open(DATA / nombre, "rb") as f:
        archivos[nombre] = client.beta.files.upload(file=f).id
    print(nombre, "→", archivos[nombre])

## 6. Abrir la sesión ☁️

Pitazo inicial. La sesión junta todo:

- `agent`: quién juega,
- `environment_id`: en qué cancha,
- `resources`: qué archivos tendrá a mano. Quedarán en la carpeta `/mnt/session/uploads/`, la misma que mencionan las instrucciones.

La sesión guarda toda la conversación en la nube. El agente recuerda lo que hablaron mientras la sesión exista.

In [ ]:
session = client.beta.sessions.create(
    agent=agent_id,
    environment_id=env_id,
    resources=[
        {"type": "file", "file_id": archivos["partidos.csv"], "mount_path": "partidos.csv"},
        {"type": "file", "file_id": archivos["goles.csv"], "mount_path": "goles.csv"},
    ],
)

session_id = session.id
print("session_id:", session_id)

## 7. Programar tu herramienta (sin API)

En la sección 2.2 anotamos la jugada en la pizarra; ahora la entrenamos. Esta función es la que responde cuando el agente pide `get_proximo_partido`.

Recibe el nombre de la herramienta y sus argumentos, busca en el `calendario` que tienes en tu computadora y devuelve la respuesta como texto. Si el equipo no existe, lo dice en lugar de inventar.

In [ ]:
import json

def handle_tool(name: str, args: dict) -> str:
    if name == "get_proximo_partido":
        equipo = args["equipo"].strip().lower()
        for p in calendario:
            if equipo in (p["local"].lower(), p["visitante"].lower()):
                return json.dumps(p, ensure_ascii=False)
        return json.dumps({"error": f"no hay partido programado para {args['equipo']}"}, ensure_ascii=False)
    return f"herramienta desconocida: {name}"

Como es una función normal de Python, la probamos antes de que la use el agente:

In [ ]:
print(handle_tool("get_proximo_partido", {"equipo": "Halcones"}))
print(handle_tool("get_proximo_partido", {"equipo": "Real Madrid"}))

## 8. Conversar: la narración del partido

Hablar con un agente no es "pregunto y me responden". Es más parecido a escuchar la narración de un partido: van llegando **eventos**, uno tras otro, que cuentan todo lo que el agente hace.

| Evento | Lo que significa |
|---|---|
| `agent.message` | El agente te escribe |
| `agent.tool_use` | El agente usa una herramienta lista, **en la nube** |
| `agent.tool_result` | Lo que esa herramienta le devolvió |
| `agent.custom_tool_use` | El agente pide **tu** herramienta y se queda esperando |
| `session.status_idle` | El agente se detuvo: terminó (`end_turn`) o espera algo de ti (`requires_action`) |

La función `preguntar` hace tres cosas:

1. Abre la narración (`stream`) y **después** envía tu pregunta, para no perderse ninguna jugada.
2. Imprime cada evento que llega.
3. Cuando el agente pide tu herramienta, ejecuta `handle_tool` en tu computadora y le envía el resultado.

Es la celda más larga del notebook. No hace falta memorizarla: léela una vez buscando esas tres partes.

In [ ]:
def texto(content) -> str:
    return "".join(getattr(b, "text", "") for b in (content or []))


def preguntar(session_id: str, pregunta: str) -> None:
    with client.beta.sessions.events.stream(session_id) as stream:
        client.beta.sessions.events.send(
            session_id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": pregunta}]}],
        )

        for ev in stream:
            if ev.type == "agent.message":
                print(f"\n🤖 {texto(ev.content)}")

            elif ev.type == "agent.tool_use":
                print(f"\n☁️  [nube] {ev.name}: {json.dumps(ev.input, ensure_ascii=False)[:300]}")

            elif ev.type == "agent.tool_result":
                print(f"    ↳ {texto(ev.content)[:300]}")

            elif ev.type == "agent.custom_tool_use":
                print(f"\n💻 [tu computadora] {ev.name}: {json.dumps(ev.input, ensure_ascii=False)}")
                resultado = handle_tool(ev.name, ev.input)
                print(f"    ↳ {resultado}")
                client.beta.sessions.events.send(
                    session_id,
                    events=[{
                        "type": "user.custom_tool_result",
                        "custom_tool_use_id": ev.id,
                        "content": [{"type": "text", "text": resultado}],
                    }],
                )

            elif ev.type == "session.status_idle":
                print(f"\n⏸  el agente se detuvo ({ev.stop_reason.type})")
                if ev.stop_reason.type != "requires_action":
                    break

### 8.1 Primera jugada: ¿ves los archivos? ☁️

Empezamos fácil, para comprobar que los archivos llegaron. Vas a ver líneas `[nube]`: es el agente ejecutando comandos en su computadora.

La primera pregunta tarda un poco más, porque la computadora del agente tiene que encenderse.

In [ ]:
preguntar(session_id, "¿Qué archivos tienes en /mnt/session/uploads/ y cuántas filas tiene cada uno?")

### 8.2 Tarea 1: la tabla de posiciones ☁️

Ahora un trabajo de verdad. En los eventos `[nube]` vas a ver el programa de Python que el agente escribe para hacer las cuentas. Nadie se lo dictó: lo decidió él.

In [ ]:
preguntar(session_id, "Arma la tabla de posiciones de la liga después de la fecha 4.")

### 8.3 Tarea 2: goleadores y datos curiosos ☁️

Seguimos en la **misma sesión**, así que el agente recuerda lo anterior. Esta pregunta necesita el segundo archivo.

In [ ]:
preguntar(session_id, "¿Quiénes son los tres máximos goleadores? ¿Y cuál fue el partido con más goles?")

### 8.4 Tarea 3: lo que solo tú sabes ☁️

El calendario no está en los archivos. Fíjate en la línea `[tu computadora]`: es el agente pidiéndote el dato, tu función `handle_tool` respondiendo y el agente siguiendo con la respuesta.

Después tiene que razonar con la tabla que ya calculó.

In [ ]:
preguntar(session_id, "¿Contra quién juegan los Halcones en la próxima fecha, cuándo y dónde? ¿Todavía pueden salir campeones? Explica qué tendría que pasar.")

### 8.5 Tarea 4: periodista deportivo ☁️

Un agente también puede crear. Le pedimos un texto usando los datos reales del torneo.

In [ ]:
preguntar(session_id, "Escribe una crónica corta (máximo 120 palabras) de la fecha 4 para el periódico del colegio, con un titular llamativo.")

## 9. ¿Acertó el agente? (sin API)

Una regla de oro al trabajar con inteligencia artificial: **verifica**. Calculamos la tabla y los goleadores por nuestra cuenta, con los mismos datos, y los comparamos con lo que respondió el agente en 8.2 y 8.3.

In [ ]:
tabla = {e: {"PJ": 0, "PTS": 0, "GF": 0, "GC": 0} for e in plantillas}

for _, local, gl, gv, visitante in partidos:
    for equipo, favor, contra in ((local, gl, gv), (visitante, gv, gl)):
        t = tabla[equipo]
        t["PJ"] += 1
        t["GF"] += favor
        t["GC"] += contra
        t["PTS"] += 3 if favor > contra else 1 if favor == contra else 0

orden = sorted(tabla.items(), key=lambda x: (x[1]["PTS"], x[1]["GF"] - x[1]["GC"], x[1]["GF"]), reverse=True)

print("Pos Equipo      PJ  PTS  GF  GC  DIF")
for pos, (equipo, t) in enumerate(orden, start=1):
    print(f'{pos:>2}  {equipo.ljust(10)} {t["PJ"]:>3} {t["PTS"]:>4} {t["GF"]:>3} {t["GC"]:>3} {t["GF"] - t["GC"]:>+4}')

In [ ]:
from collections import Counter

print("Goleadores:")
for (jugador, equipo), n in Counter((j, e) for _, e, j, _ in goles).most_common(3):
    print(f"  {n} goles · {jugador} ({equipo})")

fecha, local, gl, gv, visitante = max(partidos, key=lambda p: p[2] + p[3])
print(f"\nPartido con más goles: {local} {gl}-{gv} {visitante} (fecha {fecha})")

¿Coinciden? Si el agente siguió las instrucciones y calculó con un programa, deberían coincidir. Si alguna vez no coinciden, ya sabes cuál revisar primero: mira en los eventos `[nube]` qué programa escribió.

## 10. Limpieza ☁️

Final del partido. Las sesiones ocupan recursos en la nube, así que se borran al terminar. Borrar la sesión borra también la conversación.

El agente, el entorno y los archivos se pueden reutilizar: con ellos puedes abrir otra sesión (sección 6) sin repetir lo anterior. Las líneas con `#` los eliminan cuando ya no los quieras.

In [ ]:
client.beta.sessions.delete(session_id)
print("sesión borrada:", session_id)

# client.beta.agents.archive(agent_id)
# client.beta.environments.delete(env_id)
# for file_id in archivos.values():
#     client.beta.files.delete(file_id)

## Reto 🏆

Ahora te toca a ti.

**Córtale el internet.** El agente puede buscar en la web, pero aquí queremos que responda solo con los datos del torneo. En la sección 2.2 cambia `herramientas_integradas` por la versión de abajo, crea el agente de nuevo (sección 3) y comprueba en la sección 3.1 que `web_search` y `web_fetch` pasan a la lista de apagadas. Después pregúntale quién ganó el último Mundial: ¿qué responde ahora?

```python
herramientas_integradas = {
    "type": "agent_toolset_20260401",
    "default_config": {"enabled": True},
    "configs": [
        {"name": "web_search", "enabled": False},
        {"name": "web_fetch", "enabled": False},
    ],
}
```

## Lo que aprendiste

Construiste un agente completo con las cuatro piezas: **agente**, **entorno**, **sesión** y **eventos**. También viste la diferencia entre las herramientas que corren en la nube y las que corren en tu computadora.